In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("Filtered_TitleV_Permits.csv")



In [ ]:
# Parse dates 
df["Date Received"] = pd.to_datetime(df["Date Received"], errors="coerce")
df["Date Disposed"] = pd.to_datetime(df["Date Disposed"], errors="coerce")

In [ ]:
# Compute backlog (18 months rounded to 548 days)
df["processing_days"] = (df["Date Disposed"] - df["Date Received"]).dt.days
df["backlogged"] = df["processing_days"] > 548
df["Disposition Year"] = df["Date Disposed"].dt.year

In [ ]:
# Aggregate data by Year AND Region
regional_backlog = (
    df[df["backlogged"]]
    .dropna(subset=["Disposition Year", "Region"])
    .groupby(["Disposition Year", "Region"])
    .size()
    .unstack(fill_value=0) # This turns regions into individual columns for plotting
)

In [ ]:
# Classify backlogged permits by Region
regions = ["Northcentral", "Northwest", "Northeast", "Southcentral", "Southwest", "Southeast"]

plt.figure(figsize=(12, 6))
for region in regions:
    if region in regional_backlog.columns:
        plt.plot(regional_backlog.index, regional_backlog[region], marker='o', label=region)

# Visualize line chart 
plt.title("Backlogged Permits by Region (New & Renewal)")
plt.xlabel("Year")
plt.ylabel("Number of Backlogged Permits")
plt.legend(title="Regions")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Interacticve graph - Clickable points in order to see the details of the backlogged permits for that year and region
# The clicked region in the legend is the only one dispayed in the graph, while the others are hidden. 
# This allows users to focus on specific regions and analyze their backlog trends over time.
import plotly.express as px
import plotly.graph_objects as go
import anywidget
import pandas as pd
from IPython.display import display


fig = px.line(
    regional_backlog.reset_index(),
    x="Disposition Year",
    y=regions,
    markers=True,
    title="Backlogged Permits by Region (New & Renewal)",
    labels={"variable": "Region", "Disposition Year": "Year", "value": "Number of Backlogged Permits"},
    template="simple_white"
)
fig.update_layout(legend_title_text="Regions",
                  hovermode="x unified",
                  font = dict(family="Inter, Arial, sans-serif", size=12, color="#0072bc"), # company primary yellow blue for labels
                  legend=dict(itemclick="toggleothers", 
                              itemdoubleclick="toggle",
                              orientation="h",
                              y=1.1)) 

fig = go.FigureWidget(fig)

display(fig)



table_output = pd.DataFrame(columns=["Region", "Year", "Backlogged Permits"])
display(table_output)
def update_table(trace, points, selector):

    if points.point_inds:
        idx = points.point_inds[0]

        region = trace.name
        year = trace.x[idx]
        permits = trace.y[idx]

        table = pd.DataFrame({
            "Region": [region],
            "Year": [year],
            "Backlogged Permits": [permits]
        })

        display(table)

for trace in fig.data:
    trace.on_click(update_table)




: 